# 04 · One-stage anchor-free: YOLOv8 (Ultralytics)
Same frames / val / metric as 02-03. Ultralytics has no ignore concept, so ignore regions are painted gray in the
training images (`src/convert.py`). Per-epoch curves = Ultralytics validator on every 4th val frame; the reported
score = `src.eval.evaluate` on all val frames with `last.pt` (no checkpoint selection on val).
Runs: `yolo_baseline` YOLOv8s 640, 20 epochs (~1 h) → `yolo_hires` 1024 (~2 h) → `yolo_p2` + stride-4 P2 head (~3 h, writes a submission).

In [1]:
# Paths: auto-detected for Colab, Kaggle and a local clone. Override by hand if your layout differs.
import os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    ENV = "colab"
    from google.colab import drive
    drive.mount("/content/drive")                                   # outputs survive disconnects
    DATA_ROOT = Path("/content/data")                               # folder with test_images.csv
    REPO_DIR, WORK_DIR = Path("/content/Video-Detection-HW2"), Path("/content/drive/MyDrive/hw2_outputs")
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):                      # Colab also has /kaggle/input -> check env var
    ENV = "kaggle"
    DATA_ROOT = next(Path("/kaggle/input").glob("**/test_images.csv")).parent  # competition data
    REPO_DIR, WORK_DIR = Path("/kaggle/working/Video-Detection-HW2"), Path("/kaggle/working/outputs")
else:
    ENV = "local"
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_ROOT, WORK_DIR = REPO_DIR / "data", REPO_DIR / "outputs"
assert (DATA_ROOT / "test_images.csv").exists(), f"test_images.csv not found in {DATA_ROOT}"
print("env:", ENV, "| DATA_ROOT:", DATA_ROOT, "| REPO_DIR:", REPO_DIR, "| WORK_DIR:", WORK_DIR)

env: local | DATA_ROOT: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/data | REPO_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2 | WORK_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs


In [2]:
# Get the code + pinned dependencies on Kaggle / Colab (needs Internet). Locally nothing happens.
import subprocess
REPO_URL = "https://github.com/ystacy-ab/Video-Detection-HW2.git"
if ENV != "local":
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

In [3]:
# Make `src/` importable and print library versions (reproducibility)
sys.path.insert(0, str(Path(REPO_DIR).resolve()))
from src.utils import load_config, load_exp_config, seed_everything, print_versions, ensure_dir
print_versions()

python          3.12.6
numpy           2.4.6
torch           2.11.0
torchvision     0.26.0
ultralytics     8.4.208
faster_coco_eval 1.8.0
pycocotools     installed
ensemble_boxes  installed
sahi            0.14.0
cv2             5.0.0
wandb           0.31.0
device          CPU only


In [4]:
# Shared config + this notebook's experiment configs, seeds, output folders
cfg = load_config(Path(REPO_DIR) / "configs" / "base.yaml")
seed_everything(cfg["seed"])
DATA_ROOT, WORK_DIR = Path(DATA_ROOT), ensure_dir(WORK_DIR)
RES_DIR, FIG_DIR = Path(REPO_DIR) / "results", ensure_dir(Path(REPO_DIR) / "results" / "figures")
GT_VAL = DATA_ROOT / "val" / "annotations_coco.json"
EXPS = {n: load_exp_config(Path(REPO_DIR) / "configs" / f"{n}.yaml") for n in ['yolo_baseline', 'yolo_hires', 'yolo_p2']}
RUN = list(EXPS)  # experiments this session may TRAIN; remove names to split the work over sessions
print("experiments:", RUN)

experiments: ['yolo_baseline', 'yolo_hires', 'yolo_p2']


In [5]:
# Imports, GPU check and small helpers used below
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from src.utils import CLASS_NAMES, load_predictions
from src.eval import evaluate, per_class_table, evaluate_groups, clip_groups, density_groups
from src.viz import show_grid
from src import analysis as A
pd.set_option("display.width", 180)
RUN_GPU = torch.cuda.is_available()  # training / GPU-only cells are skipped on CPU
print("GPU:", torch.cuda.get_device_name(0) if RUN_GPU else "none -> training cells will be skipped")


def have(name, split="val"):
    return (WORK_DIR / "preds" / f"{name}_{split}.pkl").exists()


def compare(names, title=None):
    """COCO metrics of saved val predictions (same evaluator for every model)."""
    rows = {n: evaluate(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), GT_VAL, verbose=False)
            for n in names if have(n)}
    cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"] + [f"AP_{c}" for c in CLASS_NAMES]
    t = pd.DataFrame(rows, index=cols).T.round(4)
    if title and len(t):
        t.to_csv(RES_DIR / f"{title}.csv")
    return t


def speed_table(names):
    """Speed / training time from results/experiments.csv (last row per experiment)."""
    e = pd.read_csv(RES_DIR / "experiments.csv")
    e = e[e["experiment"].isin(names)].drop_duplicates("experiment", keep="last")
    return e[["experiment", "val_mAP", "AP50", "AP_small", "ms_per_img", "FPS", "train_time_min"]]

from src.models_ul import prepare_ul_data, run_ul_experiment, read_curves
from src.train_tv import prepare_indices
RESULTS = {}


def run_ul(name, test=False, notes=""):
    """Train + evaluate one Ultralytics config (resumable; finished runs are only reloaded)."""
    done = (WORK_DIR / "ul" / name / "final.json").exists()
    if not done and not (RUN_GPU and name in RUN):
        print(f"{name}: skipped (no GPU or not in RUN)")
        return None
    r = run_ul_experiment(EXPS[name], DATA_YAML, DATA_ROOT, WORK_DIR, test=test and RUN_GPU, notes=notes)
    if RUN_GPU:
        torch.cuda.empty_cache()
    RESULTS[name] = r
    display(r["per_class"].set_index("class").T.round(3))
    display(r["per_clip"].round(3))
    return r


def smoke(name):
    """1 epoch on 5 % of a tiny split + final eval on 28 val frames: checks the pipeline end to end."""
    if not RUN_GPU:
        print("smoke test skipped (no GPU)")
        return
    data_smoke = prepare_ul_data(EXPS[name], DATA_ROOT, WORK_DIR, smoke=True)
    run_ul_experiment(EXPS[name], data_smoke, DATA_ROOT, WORK_DIR, smoke=True)
    torch.cuda.empty_cache()

GPU: none -> training cells will be skipped


## 1. Ultralytics dataset (CPU, ~10-20 min the first time, then reused)

In [6]:
# Train frames (step 6) + val subset for curves, ignore regions painted gray -> WORK_DIR/yolo/data_step6.yaml
DATA_YAML = prepare_ul_data(EXPS["yolo_baseline"], DATA_ROOT, WORK_DIR)
print(open(DATA_YAML).read())

[index] train: 56 clips, 4062 frames (step=6)
[index] val: 7 clips, 2846 frames (step=1)
[data] train 4062 | val per-epoch 714 | val final 2846


convert train_step6:   0%|          | 0/4062 [00:00<?, ?it/s]

[convert] train_step6: frames=4062 masked_frames=3034 boxes_kept=183736 boxes_dropped_in_ignore=2070


convert val_step4:   0%|          | 0/714 [00:00<?, ?it/s]

[convert] val_step4: frames=714 masked_frames=360 boxes_kept=28367 boxes_dropped_in_ignore=318
[convert] wrote /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs/yolo/data_step6.yaml
path: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs/yolo
train:
- /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs/yolo/train_step6/images
val: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs/yolo/val_step4/images
nc: 8
names:
  0: person
  1: bicycle
  2: car
  3: van
  4: truck
  5: tricycle
  6: bus
  7: motor



## 2. Smoke test (GPU, ~5 min)

In [7]:
# Smoke test with the heaviest config (P2 head, 1024)
smoke("yolo_p2")

smoke test skipped (no GPU)


## 3. Baseline and tuning

In [8]:
# Baseline: YOLOv8s, 640 input, Ultralytics defaults (TAL assignment, CIoU + DFL box loss)
run_ul("yolo_baseline", notes="YOLOv8s 640 defaults");

yolo_baseline: skipped (no GPU or not in RUN)


In [9]:
# Tuning 1: input resolution 640 -> 1024
run_ul("yolo_hires", notes="imgsz 1024");

yolo_hires: skipped (no GPU or not in RUN)


In [10]:
# Tuning 2 (final anchor-free model): + P2 head (stride 4); also predicts the test set -> submission
run_ul("yolo_p2", test=True, notes="imgsz 1024 + P2 head");

yolo_p2: skipped (no GPU or not in RUN)


## 4. Search range, selected settings and comparison with the anchor-based models

In [11]:
# Searched settings and results (search range = the three configs)
rows = [{"run": n, "imgsz": c["train"]["imgsz"], "head": "P2-P5" if c["model"].get("cfg") else "P3-P5",
         "epochs": c["train"]["epochs"], "batch": c["train"]["batch"]} for n, c in EXPS.items()]
display(pd.DataFrame(rows))
names = list(EXPS) + ["frcnn_tuned", "retinanet_tuned"]
display(compare(names, "yolo_comparison"))
speed_table(names)

,run,imgsz,head,epochs,batch
0,yolo_baseline,640,P3-P5,20,16
1,yolo_hires,1024,P3-P5,20,8
2,yolo_p2,1024,P2-P5,20,6


,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR,AP_person,AP_bicycle,AP_car,AP_van,AP_truck,AP_tricycle,AP_bus,AP_motor


,experiment,val_mAP,AP50,AP_small,ms_per_img,FPS,train_time_min


In [12]:
# AP by object size (COCO buckets) and by scene density (number of GT objects per frame)
names = [n for n in ["yolo_baseline", "yolo_p2", "frcnn_tuned", "retinanet_tuned"] if have(n)]
if names:
    display(compare(names, "anchor_free_by_size")[["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]])
    dens = density_groups(GT_VAL)
    t = pd.concat([evaluate_groups(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), GT_VAL, dens)
                   .assign(model=n) for n in names])
    t.to_csv(RES_DIR / "anchor_free_by_size_by_density.csv", index=False)
    display(t.pivot_table(index="model", columns="group", values="mAP", sort=False).round(4))
else:
    print("no saved predictions yet")

no saved predictions yet


In [13]:
# Recall (score >= 0.3, IoU >= 0.5) vs. object size for each model: where are small objects lost?
names = [n for n in ["yolo_baseline", "yolo_hires", "yolo_p2", "frcnn_tuned", "retinanet_tuned"] if have(n)]
if names:
    gt_val = A.gt_from_coco(GT_VAL)
    bins = [0, 8, 12, 16, 24, 32, 48, 64, 96, 1e9]
    fig, ax = plt.subplots(figsize=(7, 4))
    for n in names:
        _, g, _ = A.analyse_predictions(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), gt_val, conf=0.3)
        r = g.groupby(pd.cut(g["size"], bins), observed=False)["matched"].mean()
        ax.plot(range(len(r)), r.values, marker="o", label=n)
    ax.set_xticks(range(len(bins) - 1), [f"{int(a)}-{int(b) if b < 1e8 else '+'}" for a, b in zip(bins[:-1], bins[1:])], rotation=30)
    ax.set(xlabel="object size sqrt(w*h) [px]", ylabel="recall", ylim=(0, 1)); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.savefig(FIG_DIR / "yolo_recall_vs_size.png", dpi=120, bbox_inches="tight"); plt.show()

In [14]:
# Training curves (Ultralytics validator, every 4th val frame)
cur = read_curves(WORK_DIR, list(EXPS))
if len(cur):
    fig, ax = plt.subplots(figsize=(7, 4))
    for n, g in cur.groupby("experiment"):
        ax.plot(g["epoch"], g["metrics/mAP50-95(B)"], marker=".", label=n)
    ax.set(xlabel="epoch", ylabel="val mAP50-95 (Ultralytics)"); ax.legend(); ax.grid(alpha=.3)
    fig.savefig(FIG_DIR / "yolo_curves.png", dpi=120, bbox_inches="tight"); plt.show()
else:
    print("no results.csv yet")

no results.csv yet
